In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt                                                                                                                                                                                                                                                                                                                              [;p[;p';        ]]
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncode                                                                                                               r


In [7]:
df = pd.read_csv("heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


In [8]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]
print(X.shape)
print(y.shape)

(918, 11)
(918,)


In [9]:
X.insert(0, "0th", 1)

In [10]:
X.head()

,0th,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,1,40,M,ATA,140,289,0,Normal,172,N,0.0,Up
1,1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat
2,1,37,M,ATA,130,283,0,ST,98,N,0.0,Up
3,1,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat
4,1,54,M,NAP,150,195,0,Normal,122,N,0.0,Up


In [11]:
categorical_cols = [
    "Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"
]
numerical_cols = X.columns.difference(categorical_cols)

In [27]:
from sklearn.preprocessing import StandardScaler

# Re-define your preprocessor to scale numerical columns
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", StandardScaler(), numerical_cols) # Change 'passthrough' to StandardScaler()
    ]
)
x_encoded = preprocessor.fit_transform(X)

In [28]:

X_train, X_test, y_train, y_test = train_test_split(x_encoded, y.values, test_size=0.2, random_state=42)

theta = np.zeros(X_train.shape[1])

In [29]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def train_logistic_regression(X, y, learning_rate, iterations):
    m = len(y)
    weights = np.zeros(X.shape[1])

    for i in range(iterations):
        z = np.dot(X, weights)
        predictions = sigmoid(z)

        # Gradient Descent: theta = theta - (alpha/m) * X.T * (h - y)
        gradient = np.dot(X.T, (predictions - y)) / m
        weights -= learning_rate * gradient

    return weights

learning_rate = 0.1
iterations = 10000
final_weights = train_logistic_regression(X_train, y_train, learning_rate, iterations)

In [30]:
test_z = np.dot(X_test, final_weights)
test_probs = sigmoid(test_z)
y_pred = (test_probs >= 0.5).astype(int)

def get_confusion_matrix(actual, predicted):

    tp = np.sum((actual == 1) & (predicted == 1))
    tn = np.sum((actual == 0) & (predicted == 0))
    fp = np.sum((actual == 0) & (predicted == 1))
    fn = np.sum((actual == 1) & (predicted == 0))

    return np.array([[tn, fp], [fn, tp]])

conf_matrix = get_confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(conf_matrix)

accuracy = (conf_matrix[0,0] + conf_matrix[1,1]) / len(y_test)
print(f"\nModel Accuracy: {accuracy * 100:.2f}%")

Confusion Matrix:
[[67 10]
 [17 90]]

Model Accuracy: 85.33%
